# 1. Imports

In [27]:
import pandas as pd
from pathlib import Path
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

# 2. Funções

## 2.1. Função para pegar os eventos de uma temporada nos arquivos parquet

In [28]:
def get_season_events_parquet_file_paths(events_competition_season_folder_path):
    
    season_events_parquet_file_paths = [
        str(Path(events_competition_season_folder_path) / season_event_parquet_file) 
        for season_event_parquet_file in os.listdir(events_competition_season_folder_path) 
        if season_event_parquet_file.endswith('.parquet')
        ]
    
    return season_events_parquet_file_paths

# 3. Preparação dos dados

## 3.1. Criação da Sessão Spark

In [29]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .master("local[*]")
    .appName("feature_engineering")
    .getOrCreate()
    )

## 3.2. Criação do df para pegar os eventos de todas as partidas da temporada de 2022-2023 da Premier League

(dps pode ser interessante levar a parte do schema dos jogadores e da bola p etapa de extração)

In [30]:
# competition_id = 1 (Premier League)
# season = 2022-2023
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

In [31]:
events_competition_season_folder_path = str(Path().resolve().parent.parent / "data" / "events" / "1" / "2022-2023")

season_events_parquet_file_paths = get_season_events_parquet_file_paths(events_competition_season_folder_path)

# Criação do dataframe concatenando todos os arquivos parquet dos eventos das partidas entre as temporadas de todas as competições
df_events = spark.read.parquet(*season_events_parquet_file_paths)

df_events = df_events.withColumnsRenamed({
    "id": "eventId",
    "player.id": "eventPlayer.id",
    "player.name": "eventPlayer.name",
    "team.id": "eventTeam.id",
    "team.name": "eventTeam.name",
})

# Schema em Pyspark para poder parsear o json dos dados de tracking dos jogadores que está como string
players_schema = ArrayType(
    StructType([
        #StructField("speed", FloatType(), True),
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("player", StructType([
            #StructField("id", IntegerType(), True), 
            StructField("name", StringType(), True)]), 
            True),
        #StructField("visibility", StringType(), True),
        #StructField("confidence", StringType(), True),
        #StructField("jerseyNum", StringType(), True)      
    ])
)

# Schema em Pyspark para poder parsear o json dos dados de tracking da bola que está como string
balls_schema = ArrayType(
    StructType([
        StructField("x", FloatType(), True),
        StructField("y", FloatType(), True),
        StructField("z", FloatType(), True),
        #StructField("visibility", StringType(), True)
    ])
)

details_schema = MapType(StringType(), StringType())

df_events = df_events.withColumns({
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time mandante
    "homePlayers_parsed": F.from_json("homePlayers", players_schema),
    
    # Cria coluna com json parseado para Lista de dicionários para dados de tracking do time adversário
    "awayPlayers_parsed": F.from_json("awayPlayers", players_schema),

    # Cria coluna com json parseado para dicionário para dados de tracking da bola
    "balls_parsed": F.from_json("balls", balls_schema),

    "details_parsed": F.from_json("details", details_schema)

}).drop('homePlayers', 'awayPlayers', 'balls', 'details')

df_events = df_events.select(
    #'competitionId',
    #'season',
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    #'periodDescription',
    'startFormattedGameClock',
    'startGameClock',
    #'details_parsed',
    'homeTeam',
    #F.col('`eventPlayer.id`').alias('eventPlayerId'),
    F.col('`eventPlayer.name`').alias('eventPlayerName'),
    #F.col('`eventTeam.id`').alias('eventTeamId'), 
    F.col('`eventTeam.name`').alias('eventTeamName'), 
    'homePlayers_parsed', 
    'awayPlayers_parsed', 
    'balls_parsed'
)

In [32]:
print('Quantidade de linhas:', df_events.count())

Quantidade de linhas: 945154


## 3.2. Obter jogos da temporada e ajustar identificação do mandante/adversário

(dps pode ser interessante levar isso p etapa de extração)

In [33]:
games_path = str(Path().resolve().parent.parent / "data" / "games.csv")

df_games = spark.read.csv(games_path, header=True)

df_games_raw = df_games.withColumnRenamed("id","gameId").filter(F.col('season') == '2022-2023')

#df_games_raw.show()

In [34]:
df_games_raw.select('venueType').distinct().show()

+-------------+
|    venueType|
+-------------+
|      NEUTRAL|
|OPPONENT_HOME|
|    TEAM_HOME|
+-------------+



In [35]:
df_games_raw.filter(F.col('venueType') == 'NEUTRAL').select('gameId', 'date', 'season', 'venueType', '`team.name`', '`opponentTeam.name`', '`stadium.name`').sort('date').show(truncate=False)

+------+----------+---------+---------+----------------------+-----------------------+-------------+
|gameId|date      |season   |venueType|team.name             |opponentTeam.name      |stadium.name |
+------+----------+---------+---------+----------------------+-----------------------+-------------+
|4443  |2022-08-06|2022-2023|NEUTRAL  |Chelsea               |Everton                |Goodison Park|
|4458  |2022-08-20|2022-2023|NEUTRAL  |Everton               |Nottingham Forest      |Goodison Park|
|4490  |2022-09-03|2022-2023|NEUTRAL  |Everton               |Liverpool              |Goodison Park|
|4510  |2022-09-18|2022-2023|NEUTRAL  |Everton               |West Ham               |Goodison Park|
|4531  |2022-10-09|2022-2023|NEUTRAL  |Everton               |Manchester United      |Goodison Park|
|4558  |2022-10-22|2022-2023|NEUTRAL  |Crystal Palace        |Everton                |Goodison Park|
|4578  |2022-11-05|2022-2023|NEUTRAL  |Everton               |Leicester City         |Goodi

- Apesar de Goodison Park teoricamente ser estádio do Everton e se tratarem de todos os jogos do Everton, o venueType é neutro. Por conta disso, não irei considerar essas partidas.
- Caso necessário usá-las futuramente, podemos ver como ficam os eventos de home e away e se seguem essa estrutura acima mesmo o estádio sendo neutro. Uma sugestão pode ser considerar sempre Everton como casa, mas teria que ver se os eventos de posse ficam de acordo.

In [36]:
df_games_raw = df_games_raw.filter(F.col('venueType').isin(['TEAM_HOME', 'OPPONENT_HOME']))

In [37]:
# nesse jogo o liverpool começa do lado direito atacando para a esquerda (https://www.youtube.com/watch?v=Mh82yD4YT6A)
df_games_raw.filter(F.col('gameId') == 4541).show(5)

# nesse jogo o manchester city começa no lado esquerdo atacando para a direita (https://www.youtube.com/watch?v=G1JQc5F-w_g)
df_games_raw.filter(F.col('gameId') == 4452).show(5)

+------+----------+---------+----------------------+-------------+---------+-------+---------+--------------+----------------+---------------+-----------------+------------+--------------+-------------+
|gameId|      date|   season|teamExtraTimeStartSide|teamStartSide|venueType|team.id|team.name|competition.id|competition.name|opponentTeam.id|opponentTeam.name|stadium.name|stadium.length|stadium.width|
+------+----------+---------+----------------------+-------------+---------+-------+---------+--------------+----------------+---------------+-----------------+------------+--------------+-------------+
|  4541|2022-10-16|2022-2023|                 Right|        Right|TEAM_HOME|     10|Liverpool|             1|  Premier League|             11|  Manchester City|     Anfield|         101.0|         68.0|
+------+----------+---------+----------------------+-------------+---------+-------+---------+--------------+----------------+---------------+-----------------+------------+--------------+

- A variável teamStartSide se refere sempre ao team.name. A questão é saber se esse lado se refere ao lado que o time começa a partida ou se é o sentido do ataque.

- No primeiro jogo (ID 4541), o Liverpool está jogando em casa e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.
- No segundo jogo (ID 4452), o AFC Bournemouth está jogando na casa do adversário Manchester City e inicia do lado direito, ou seja, o sentido do seu ataque é para a esquerda.

- Logo, teamStartSide se refere ao lado no campo que o time começa e o sentido do ataque é na direção oposta (teamStartSide = 'Right', então AttackDirection = 'Left')

In [38]:
# se venueType == TEAM_HOME, (homeTeam.id == team.id e homeTeam.name == team.name) e (opponentTeam.id == opponentTeam.id e opponentTeam.name == opponentTeam.name)
# se venueType == OPPONENT_HOME, (homeTeam.id == opponentTeam.id e homeTeam.name == opponentTeam.name) e (opponentTeam.id == team.id e opponentTeam.name == team.name)
df_games = (
    df_games_raw
    .withColumns({
        # homeTeam = "team" quando o mandante é o "team" (TEAM_HOME), senão homeTeam = "opponentTeam"
        "homeTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.id`")).otherwise(F.col("`opponentTeam.id`")),
        "homeTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`team.name`")).otherwise(F.col("`opponentTeam.name`")),
        # homeTeamStartSide = lado que o time mandante começou:
        # - venueType == TEAM_HOME: mandante é o "team" -> usa teamStartSide direto
        # - venueType == OPPONENT_HOME: mandante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide (Right vira Left e vice-versa)
        "homeTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", F.col("teamStartSide")
            ).otherwise(
                F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))),
        
        # opponentTeam = "opponentTeam" quando o mandante é o "team" (TEAM_HOME), senão opponentTeam = "team"
        "opponentTeamId": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.id`")).otherwise(F.col("`team.id`")),
        "opponentTeamName": F.when(F.col("venueType") == "TEAM_HOME", F.col("`opponentTeam.name`")).otherwise(F.col("`team.name`")),
        # opponentTeamStartSide = lado que o time visitante começou:
        # - venueType == TEAM_HOME: visitante é o "opponentTeam" (lado não vem direto na base) ->
        #   usa o complementar do teamStartSide
        # - venueType == OPPONENT_HOME: visitante é o próprio "team" -> usa teamStartSide direto
        "opponentTeamStartSide": F.when(
            F.col("venueType") == "TEAM_HOME", 
            F.when(F.col("teamStartSide") == "Right", F.lit("Left")).otherwise(F.lit("Right"))
            ).otherwise(F.col("teamStartSide")),
    })
    .select(
        'gameId',
        F.col('`competition.id`').alias('competitionId'),
        F.col('`competition.name`').alias('competitionName'), 
        'date',
        'season',
        'venueType',
        #'homeTeamId',
        'homeTeamName',
        #'opponentTeamId',
        'opponentTeamName',
        'homeTeamStartSide',
        'opponentTeamStartSide',
        F.col('`stadium.name`').alias('stadiumName'), 
        F.col('`stadium.length`').cast("float").alias('stadiumLength'), 
        F.col('`stadium.width`').cast("float").alias('stadiumWidth')
    )
)

#df_games.show(5)

## 4. Junção dos dados dos Jogos + Eventos em uma tabela

In [39]:
# left join dos eventos + informações dos jogos
df_games_events = (
    df_events
    .join(
        df_games, 
        on = "gameId", 
        how='left'
    )
    # filtro para remover os jogos que tinha mandante neutro (venueType == NEUTRAL)
    .filter(~F.col('date').isNull())
)

df_games_events = (
    df_games_events
    # considerar a reversão de lado conforme mudança do primero para o segundo tempo
    # se for primeiro tempo, mantém a variável de StartSide, se não é o contrário
    .withColumns({
        'homeTeamStartSide': F.when(F.col('period') == 1, F.col('homeTeamStartSide')).otherwise(F.col('opponentTeamStartSide')),
        'opponentTeamStartSide': F.when(F.col('period') == 1, F.col('opponentTeamStartSide')).otherwise(F.col('homeTeamStartSide'))
    }) 

    # Sentido do ataque do time é sempre o lado que o outro time começou o período
    .withColumns({
        'homeTeamAttackDirection': F.col('opponentTeamStartSide'),
        'awayTeamAttackDirection': F.col('homeTeamStartSide')
    })
    #.drop('homeTeamStartSide')
)

#df_games_events.show(5)

In [40]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4452)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4452)).select(filter_cols).show(1)

+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|   homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|  4452|     1|Manchester City| AFC Bournemouth|             Left|                  Right|                Right|                   Left|
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
only showing top 1 row
+------+------+---------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|   homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+---

In [41]:
filter_cols = ['gameId', 'period', 'homeTeamName', 'opponentTeamName', 'homeTeamStartSide', 'homeTeamAttackDirection', 'opponentTeamStartSide', 'awayTeamAttackDirection']

df_games_events.filter((F.col('period') == 1) & (F.col('gameId') == 4541)).select(filter_cols).show(1)
df_games_events.filter((F.col('period') == 2) & (F.col('gameId') == 4541)).select(filter_cols).show(1)

+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|  4541|     1|   Liverpool| Manchester City|            Right|                   Left|                 Left|                  Right|
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
only showing top 1 row
+------+------+------------+----------------+-----------------+-----------------------+---------------------+-----------------------+
|gameId|period|homeTeamName|opponentTeamName|homeTeamStartSide|homeTeamAttackDirection|opponentTeamStartSide|awayTeamAttackDirection|
+------+------+------------+-----------

- Os times mandante e adversário, seus lados na partida e seus sentidos de ataque parecem ter sido definidos corretamente.

In [42]:
df_games_events.groupby('period').count().show()

+------+------+
|period| count|
+------+------+
|     1|456103|
|     2|442797|
+------+------+



In [43]:
# variável que indica o time com a posse
df_games_events.groupBy('homeTeam').count().show()

+--------+------+
|homeTeam| count|
+--------+------+
|    NULL|  6538|
|    true|451847|
|   false|440515|
+--------+------+



In [44]:
# window function pra criação do Id de posse
w_pos = (
    Window
    .partitionBy(
        "competitionId",
        "season",
        "gameId"
    )
    .orderBy("startGameClock")
)

df_games_events = (
    df_games_events
    .filter(
        # filtro para garantir apenas eventos das partidas no 1º e 2º tempo
        (F.col('period').isin([1,2])) &
        # filtro para não considerar tracking da bola e jogadores home/away que n tem tracking (dps podemos pensar em imputar)
        (F.size("balls_parsed") != 0) & (F.size("homePlayers_parsed") != 0) & (F.size("awayPlayers_parsed") != 0)
    ) 
    .dropna(subset='homeTeam') # drop nos eventos onde nenhum dos dois times tem a posse
    .withColumn(
        "possession_id",
        F.sum(
            F.when(
                F.col("homeTeam") != F.lag("homeTeam").over(w_pos), 1
            ).otherwise(0)
        ).over(
            w_pos.rowsBetween(Window.unboundedPreceding, Window.currentRow)
        )
    )
)

## Normalização do ataque sempre pra direita

In [45]:
# time com a posse está atacando e time sem está defendendo
df_games_events_tracking = (
    df_games_events
    # time atacando = se o time da casa tiver a posse, pega tracking home, se não pega tracking away
    .withColumn(
        "attackingPlayers",
        F.when(F.col("homeTeam"), F.col("homePlayers_parsed"))
        .otherwise(F.col("awayPlayers_parsed"))
    )
    # time defendendo = se o time da casa tiver a posse, pega tracking away, se não pega tracking home
    .withColumn(
        "defendingPlayers",
        F.when(F.col("homeTeam"), F.col("awayPlayers_parsed"))
        .otherwise(F.col("homePlayers_parsed"))
    )
    .withColumn(
        "attackingDirection",
        F.when(F.col("homeTeam"), F.col("homeTeamAttackDirection"))
        .otherwise(F.col("awayTeamAttackDirection"))
    )
    # flag para normalização (para tratar ataque sempre pra direita)
    .withColumn(
        "need_side_revert",
        F.col("attackingDirection") == "Left"
    )
    .drop(
        'homePlayers_parsed',  
        'awayPlayers_parsed',
        'homeTeamAttackDirection',
        'awayTeamAttackDirection'
    )
)

#df_games_events_tracking.show()

In [46]:
players_tracking_norm = (
        lambda p: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -p["x"])
            .otherwise(p["x"])
            .alias("x"),
            # variáveis restantes mantém igual
            p["y"].alias("y"),            
            p["player"].alias("player"),
            #p["visibility"].alias("visibility"),
            #p["confidence"].alias("confidence")
        )
)

balls_norm = (
    F.transform(
        "balls_parsed",
        lambda b: F.struct(
            # reversão do eixo horizontal qnd necessário
            F.when(F.col("need_side_revert"), -b["x"])
            .otherwise(b["x"])
            .alias("x"),
            # variáveis restantes mantém igual            
            b["y"].alias("y"),
            b["z"].alias("z"),            
            #b["visibility"].alias("visibility")
        )
    )
)

# df para normalizar os atacantes, defensores e bola sempre atacando do lado direito
df_games_events_tracking_norm = (
    df_games_events_tracking
    # normalizar atacantes
    .withColumns({
        "attackingPlayersNorm": F.transform("attackingPlayers", players_tracking_norm),
        # normalizar defensores
        "defendingPlayersNorm": F.transform("defendingPlayers", players_tracking_norm),
        # normalizar a bola
        "ballsNorm": balls_norm
    }).drop(
        'attackingPlayers', 
        'defendingPlayers',
        'balls_parsed',
        #'attackingDirection',
        #'need_side_revert'
        )
)

#df_games_events_tracking_norm.show(5)

In [47]:
def euclidean_dist(x1, y1, x2, y2):
    return F.sqrt(
        F.pow(x1 - x2, 2) +
        F.pow(y1 - y2, 2)
    )

In [48]:
# Extremos esquerdo e direito do campo no plano cartesiano
left_x = -F.col('stadiumLength') / 2
right_x = F.col('stadiumLength') / 2

# Extremos superior e inferior do campo no plano cartesiano
top_y = F.col('stadiumWidth') / 2
bottom_y = -F.col('stadiumWidth') / 2

# Eixos x e y da posição da bola a partir dos dados de tracking
ball_x = F.get("ballsNorm", 0)["x"]
ball_y = F.get("ballsNorm", 0)["y"]

## Criação das componentes de ameaça

- Métrica 1: Distância percorrida no campo (medida pela menor distância entre os escanteios do time com a posse até a bola) 
    - Hipótese: Quanto mais o jogador com posse percorrer o campo com a bola na direção do gol, maior a ameaça de gol por estar mais próximo dele.
    - Relação: Diretamente proporcional
- Métrica 2: Quantidade total de jogadores dos dois times (entre a bola e o gol)
    - Hipótese: Quanto MAIS jogadores dos dois times entre a bola e gol, MENOR a ameaça de gol por haver maior possibilidade de alguma ação defensiva e também por haver chances de um possível chute ser bloqueado.
    - Relação: Inversamente proporcional
- Métrica 3: Vantagem numérica do ataque em relação à defesa (entre a bola e o gol)
    - Hipótese: Quanto MAIOR a vantagem numérica do ataque em relação à defesa, MAIOR a ameaça de gol por ter maiores chance de ações ofensivas e menores chances de ações defensivas
    - Relação: Diretamente proporcional

In [49]:
def plot_threat_event(df_threat, show_player_names=False):
    """
    Plota um evento para validar:
    - attackers_between_ball_goal[_zona]
    - defenders_between_ball_goal[_zona]
    - total_players[_zona]
    - atk_def_advantage[_zona]

    Espera um DataFrame Spark contendo exatamente um evento.

    Sempre desenha: linha da bola, linha do meio-campo (x=0) e as
    2 linhas que dividem o campo em 3 terços.
    """

    pdf = df_threat.toPandas()

    if len(pdf) != 1:
        raise ValueError("O dataframe deve conter exatamente um evento.")

    row = pdf.iloc[0]

    attackers = row["attackingPlayersNorm"]
    defenders = row["defendingPlayersNorm"]
    ball = row["ballsNorm"][0]

    stadium_length = row["stadiumLength"]
    stadium_width = row["stadiumWidth"]

    left_x = -stadium_length / 2
    right_x = stadium_length / 2
    bottom_y = -stadium_width / 2
    top_y = stadium_width / 2

    ball_x = ball["x"]

    fig = go.Figure()

    # ============================
    # Atacantes
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in attackers],
        y=[p["y"] for p in attackers],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in attackers] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "red" if p["x"] >= ball_x else "lightcoral"
                for p in attackers
            ]
        ),
        name="Attackers"
    )
)

    # ============================
    # Defensores
    # ============================

    fig.add_trace(
    go.Scatter(
        x=[p["x"] for p in defenders],
        y=[p["y"] for p in defenders],
        mode="markers+text" if show_player_names else "markers",
        text=[p["player"]["name"] for p in defenders] if show_player_names else None,
        textposition="top center",
        marker=dict(
            size=10,
            color=[
                "blue" if p["x"] >= ball_x else "lightblue"
                for p in defenders
            ]
        ),
        name="Defenders"
    )
)
    
    # ============================
    # Bola
    # ============================

    fig.add_trace(
        go.Scatter(
            x=[ball["x"]],
            y=[ball["y"]],
            mode="markers",
            marker=dict(
                color="black",
                size=10,
                symbol="circle"
            ),
            name="Ball"
        )
    )

    # ============================
    # Linha da bola
    # ============================

    fig.add_vline(
        x=ball_x,
        line_dash="dash",
        line_width=2,
        line_color="black"
    )

    # ============================
    # Linhas divisórias fixas: meio-campo + 2 terços
    # ============================

    field_length = right_x - left_x

    zone_boundaries = [
        0.0,                                # meio-campo (half)
        left_x + field_length / 3,          # início do terço 2
        left_x + 2 * field_length / 3,      # início do terço 3
    ]

    for x_boundary in zone_boundaries:
        fig.add_vline(
            x=x_boundary,
            line_dash="dot",
            line_width=2,
            line_color="gray"
        )
    
    # ============================
    # Limites do campo
    # ============================

    fig.update_xaxes(
        range=[left_x, right_x],
        title="X",
        zeroline=False
    )

    fig.update_yaxes(
        range=[bottom_y, top_y],
        title="Y",
        scaleanchor="x",
        scaleratio=1,
        zeroline=False
    )
    
    # ============================
    # Layout
    # ============================
    height = 600
    width = int(height * stadium_length / stadium_width)
    
    fig.update_layout(
    template="simple_white",
    width=width,
    height=height,
    margin=dict(l=20, r=120, t=60, b=20),
    title=(
        f"Attacking: {row['eventTeamName']} | "
        f"Event: {row['eventTypeDescription']} | "
        #f"Total: {row['total_players']} | "
        #f"Advantage: {row['atk_def_advantage']} | "
        #f"Progression: {row['progression_dist']} | "
        #f"Threat: {row['threat_score']:.3f}"
    ),
    legend=dict(
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        orientation="v",
        bgcolor="rgba(255,255,255,0.8)"
    )
    )

    fig.show()

In [50]:
# ============================================================
# Funções reutilizáveis para cálculo por zona do campo (eixo x)
# ============================================================

# Cada zona é definida só pelo x_start (ponto de corte) — sempre [x_start, right_x].
# 'full' usa x_start = left_x (equivale a nunca mascarar, já que ball_x >= left_x sempre).
ZONES = ['full', 'half', 'third_2', 'third_3']


def get_zone_x_start(zone, left_x=left_x, right_x=right_x):
    """
    Retorna o x_start (ponto de corte) de onde a zona passa a valer.
    A zona é sempre [x_start, right_x] — jogadores/distância considerados
    dali até o gol.
    """
    field_length = right_x - left_x

    if zone == 'full':
        return left_x
    elif zone == 'half':
        return F.lit(0.0)
    elif zone == 'third_2':
        return left_x + field_length / 3
    elif zone == 'third_3':
        return left_x + 2 * field_length / 3
    else:
        raise ValueError(f"Zona inválida: {zone}")


def zone_col_names(zone):
    """Padroniza os nomes de coluna de cada zona (zona 'full' não leva sufixo)."""
    suffix = '' if zone == 'full' else f'_{zone}'
    return {
        'attackers':   f'attackers_between_ball_goal{suffix}',
        'defenders':   f'defenders_between_ball_goal{suffix}',
        'total':       f'total_players{suffix}',
        'advantage':   f'atk_def_advantage{suffix}',
        'progression': f'progression_dist{suffix}',
    }


def add_zone_metrics(df, ball_x, ball_y, right_x, zones=ZONES):
    """
    Para cada zona [x_start, right_x]:
      - se a bola estiver dentro (ball_x >= x_start): calcula total_players,
        atk_def_advantage e progression_dist normalmente (a partir de x_start)
      - se a bola estiver fora (ball_x < x_start): NULL

    Mesma função para todas as zonas, incluindo 'full' (que nunca mascara,
    pois ball_x >= left_x é sempre verdadeiro).
    """
    for zone in zones:
        x_start = get_zone_x_start(zone)
        cols = zone_col_names(zone)

        ball_in_zone = ball_x >= x_start

        attackers = F.size(
            F.filter(
                F.col('attackingPlayersNorm'),
                lambda p: (p['x'] <= right_x) & (p['x'] >= ball_x)
            )
        )
        defenders = F.size(
            F.filter(
                F.col('defendingPlayersNorm'),
                lambda p: (p['x'] <= right_x) & (p['x'] >= ball_x)
            )
        )
        total = attackers + defenders
        advantage = attackers - defenders

        progression = F.round(
            F.least(
                euclidean_dist(ball_x, ball_y, x_start, top_y),
                euclidean_dist(ball_x, ball_y, x_start, bottom_y)
            ), 2
        )

        df = df.withColumns({
            cols['attackers']:   F.when(ball_in_zone, attackers).otherwise(F.lit(None)),
            cols['defenders']:   F.when(ball_in_zone, defenders).otherwise(F.lit(None)),
            cols['total']:       F.when(ball_in_zone, total).otherwise(F.lit(None)),
            cols['advantage']:   F.when(ball_in_zone, advantage).otherwise(F.lit(None)),
            cols['progression']: F.when(ball_in_zone, progression).otherwise(F.lit(None)),
        })

    return df


df_games_events_players_ball_goal = add_zone_metrics(
    df_games_events_tracking_norm, ball_x, ball_y, right_x
)

### Criação da Ameaça pela Média das 3 componentes normalizadas com Min-Max

In [51]:
# ============================================================
# Normalização Min-Max para cada zona (full, half, third_2, third_3)
# ============================================================

# Lista de (nome_da_coluna, inverter) para cada zona:
# - progression: diretamente proporcional (mais distância percorrida = mais ameaça)
# - total: inversamente proporcional (mais jogadores entre bola e gol = menos ameaça)
# - advantage: diretamente proporcional (mais vantagem numérica do ataque = mais ameaça)
norm_targets = []
for zone in ZONES:
    cols = zone_col_names(zone)
    norm_targets.append((cols['progression'], False))  # diretamente proporcional
    norm_targets.append((cols['total'], True))          # inversamente proporcional
    norm_targets.append((cols['advantage'], False))     # diretamente proporcional

# Monta os agregados de min/max de cada coluna acima (uma única passada no df)
agg_exprs = []
for col_name, _ in norm_targets:
    agg_exprs.append(F.min(col_name).alias(f'min_{col_name}'))
    agg_exprs.append(F.max(col_name).alias(f'max_{col_name}'))

# Executa a agregação (F.min/F.max ignoram os NULLs do mascaramento por zona)
stats = df_games_events_players_ball_goal.agg(*agg_exprs).first()

# Aplica a fórmula min-max (x - min) / (max - min) em cada coluna,
# invertendo (1 - minmax) quando a métrica for inversamente proporcional
norm_cols = {}
for col_name, invert in norm_targets:
    min_val = stats[f'min_{col_name}']
    max_val = stats[f'max_{col_name}']
    expr = (F.col(col_name) - F.lit(min_val)) / F.lit(max_val - min_val)
    if invert:
        expr = 1 - expr
    norm_cols[f'{col_name}_norm'] = F.round(expr, 3)

df_threat = df_games_events_players_ball_goal.withColumns(norm_cols)

# Threat score por zona = média das 3 componentes normalizadas daquela zona
threat_score_cols = {}
for zone in ZONES:
    cols = zone_col_names(zone)
    suffix = '' if zone == 'full' else f'_{zone}'
    threat_score_cols[f'threat_score{suffix}'] = F.round(
        (
            F.col(f"{cols['progression']}_norm") +
            F.col(f"{cols['total']}_norm") +
            F.col(f"{cols['advantage']}_norm")
        ) / F.lit(3.0), 3
    )

df_threat_final = df_threat.withColumns(threat_score_cols)

# Checkpoint para truncar a lineage antes de seguir com as próximas análises
df_threat_final = df_threat_final.localCheckpoint()

In [52]:
df_threat_final.show()

+------+--------------------+------------+--------------------+------+-----------------------+--------------+--------+----------------+--------------+-------------+---------------+----------+---------+-------------+--------------+----------------+-----------------+---------------------+-------------+-------------+------------+-------------+------------------+----------------+--------------------+--------------------+--------------------+---------------------------+---------------------------+-------------+-----------------+----------------+--------------------------------+--------------------------------+------------------+----------------------+---------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+------------------------+---------------------+------------

In [53]:
threat_cols = [
    'date',
    'season',
    'gameId',
    'eventId',
    'eventType',
    'eventTypeDescription',
    'period',
    'startFormattedGameClock',
    'startGameClock',
    'homeTeam',
    'competitionName',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'progression_dist_norm',
    'total_players_norm',
    'atk_def_advantage_norm',
    'progression_dist_half_norm',
    'total_players_half_norm',
    'atk_def_advantage_half_norm',
    'progression_dist_third_2_norm',
    'total_players_third_2_norm',
    'atk_def_advantage_third_2_norm',
    'progression_dist_third_3_norm',
    'total_players_third_3_norm',
    'atk_def_advantage_third_3_norm',
    'threat_score',
    'threat_score_half',
    'threat_score_third_2',
    'threat_score_third_3'
    
]

output_path = str(Path().resolve().parent.parent / "data" / "threat_dataset")
df_threat_final.select(threat_cols).write.mode("overwrite").parquet(output_path)

In [54]:
# ============================================================
# Validação: chutes do lado errado do campo após normalização
# ============================================================
# Com o ataque sempre normalizado para a direita, um chute (eventTypeDescription == 'Shot')
# só faz sentido teoricamente com a bola já relativamente perto do gol direito.
# Se ball_x < 30, a bola ainda estaria longe demais do gol pra um chute fazer sentido —
# indica que a normalização do lado do ataque (need_side_revert) falhou para esse evento.

shot_events = (
    df_threat_final
    .filter(
        (F.col('eventTypeDescription') == 'Shot')
    )
)

shots_wrong_side = (
    shot_events
    .filter(ball_x < 20)
)

print(
    f'Quantidade de event_id de chutes do lado errado: {shots_wrong_side.select('eventId').distinct().count()}/{shot_events.count()}'
)

shots_wrong_side.select(
    'gameId',
    'eventId',
    ball_x.alias('ball_x'),
    'eventTypeDescription',
    'homeTeam',
    'homeTeamName',
    'opponentTeamName'
).show(50, truncate=False)

Quantidade de event_id de chutes do lado errado: 39/9456
+------+--------------------------------+------+--------------------+--------+-----------------------+-----------------------+
|gameId|eventId                         |ball_x|eventTypeDescription|homeTeam|homeTeamName           |opponentTeamName       |
+------+--------------------------------+------+--------------------+--------+-----------------------+-----------------------+
|4505  |cf0cb47ac8d6d86a34ccda0f4d1a15eb|0.43  |Shot                |false   |West Ham               |Newcastle United       |
|4505  |c63055d30bafefd200785162dfb761df|14.49 |Shot                |false   |West Ham               |Newcastle United       |
|4622  |b7d557860efcedcfb5d3768421561f07|11.55 |Shot                |false   |Leeds United           |West Ham               |
|4626  |6b690674521829b8ceb5fdc8e1892fb3|13.34 |Shot                |true    |Aston Villa            |Leeds United           |
|4442  |e6d3e1bf50bbb3bb87e892450a584b7b|-9.22 |Shot  

In [55]:
df_event = df_threat_final.filter(F.col("eventId") == 'e6d3e1bf50bbb3bb87e892450a584b7b')

plot_threat_event(df_event, show_player_names=False)

In [56]:
# validar eventos defensivos sendo tb do time atacando
# possession ids 289-291